# NDA dynamic spectrum

This notebook plots the Nançay Decameter Array (NDA) heliograph spectrograph
data. NDA covers **10 - 80 MHz** and records two channels which correspond to
left and right circular polarisations. The FITS file structure exposes:

- `HDU[0]` primary header (DATE-OBS, TIME-OBS, TIME-END)
- `HDU[1]` channel 1 data (n_time x n_freq)
- `HDU[2]` channel 2 data (n_time x n_freq)
- `HDU[3]` frequency axis (MHz)
- `HDU[4]` time axis (seconds since start)

**Workflow:** load FITS &rarr; build (time, freq) DataFrames for both
channels &rarr; subtract a per-channel median background &rarr; plot both
channels and their difference.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from matplotlib.ticker import AutoMinorLocator
from matplotlib.colors import LogNorm
import matplotlib as mpl

# Use the precise matplotlib epoch (avoids ~10 us offsets in old matplotlib).
mpl.rcParams['date.epoch'] = '1970-01-01T00:00:00'
try:
    mdates.set_epoch('1970-01-01T00:00:00')
except RuntimeError:
    pass

# Unified plotting style for all dynamic spectra notebooks.
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11

from astropy.io import fits
from datetime import datetime

## Configuration


In [ ]:
data_dir = './sample_data/nda'
outputs  = './outputs'
os.makedirs(outputs, exist_ok=True)

# Pick the first NDA FITS file in the data directory.
nda_files = sorted(glob.glob(f'{data_dir}/NDA_*.fits'))
print(*nda_files, sep='\n')
filename = nda_files[0]


## Helper functions


In [ ]:
def subtract_background_median(df):
    """
    Subtract a per-channel median background from a dynamic spectrum.

    The function computes the median along the time axis (axis=0) for each
    frequency channel, then subtracts it from every time sample of that
    channel. This is the standard approach for highlighting transient
    emission against a slowly-varying instrumental/sky background.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame with shape (n_times, n_freqs). Index is time, columns
        are frequencies.

    Returns
    -------
    pandas.DataFrame
        Same shape as input with the per-channel median removed.
    """
    bkg = np.nanmedian(df.values, axis=0)
    return df - np.tile(bkg, (df.shape[0], 1))


## Load NDA data


In [ ]:
nda = fits.open(filename)
nda.info()

# Frequency axis in MHz and channel arrays of shape (n_time, n_freq).
nda_freq = np.asarray(nda[3].data).reshape(-1)
nda_ch1  = np.asarray(nda[1].data)
nda_ch2  = np.asarray(nda[2].data)

# Build the absolute time axis from the start/end timestamps in the header.
start_str = nda[0].header['DATE-OBS'] + ' ' + nda[0].header['TIME-OBS']
end_str   = nda[0].header['DATE-OBS'] + ' ' + nda[0].header['TIME-END']
start_obs = datetime.strptime(start_str, '%d/%m/%Y %H:%M:%S')
end_obs   = datetime.strptime(end_str,   '%d/%m/%Y %H:%M:%S')

time_axis = pd.date_range(start=start_obs, end=end_obs, periods=nda_ch1.shape[0])

df_ch1 = pd.DataFrame(nda_ch1, index=time_axis, columns=nda_freq)
df_ch2 = pd.DataFrame(nda_ch2, index=time_axis, columns=nda_freq)

print(f'Shape (time, freq): {df_ch1.shape}')
print(f'Time range: {df_ch1.index[0]} -> {df_ch1.index[-1]}')
print(f'Freq range: {nda_freq[0]:.2f} - {nda_freq[-1]:.2f} MHz')


## Remove the per-channel median background


In [ ]:
df_ch1_nobkg = subtract_background_median(df_ch1)
df_ch2_nobkg = subtract_background_median(df_ch2)
df_diff      = df_ch2_nobkg - df_ch1_nobkg


## Plot both channels and their difference


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=True)

panels = [
    (df_ch1_nobkg, 'NDA channel 1 (background subtracted)'),
    (df_ch2_nobkg, 'NDA channel 2 (background subtracted)'),
    (df_diff,      'NDA channel 2 - channel 1'),
]

for ax, (df, title) in zip(axes, panels):
    vmin = np.nanpercentile(df.values, 5)
    vmax = np.nanpercentile(df.values, 99)
    pc = ax.pcolormesh(df.index, df.columns, df.T,
                       vmin=vmin, vmax=vmax, cmap='Spectral_r')
    fig.colorbar(pc, ax=ax, pad=0.02, label='Intensity (a.u.)')
    ax.set_ylabel('Frequency (MHz)')
    ax.set_title(title)
    ax.set_ylim(ax.get_ylim()[::-1])
    ax.yaxis.set_minor_locator(AutoMinorLocator(n=10))

axes[-1].set_xlabel('Time (UT)')
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
fig.tight_layout()

date_tag = start_obs.strftime('%Y-%m-%d')
fig.savefig(f'{outputs}/nda_dyspec_{date_tag}.png', bbox_inches='tight')
plt.show()
